# PLDDT Analysis for Priority Unstudied Genes (PUGs)

This notebook fetches PLDDT (Predicted Local Distance Difference Test) metrics from the AlphaFold API for each UniProt ID in the priority unstudied genes dataset.

## Workflow:
1. Load priority unstudied genes data
2. Extract UniProt IDs 
3. Query AlphaFold API for PLDDT metrics
4. Save results to intermediate files directory

## Output Metrics:
- `modelEntityId`: UniProt identifier used in AlphaFold
- `globalMetricValue`: Overall confidence score
- `fractionPlddtVeryLow`: Fraction with very low confidence (0-50)
- `fractionPlddtLow`: Fraction with low confidence (50-70)
- `fractionPlddtConfident`: Fraction with confident predictions (70-90)
- `fractionPlddtVeryHigh`: Fraction with very high confidence (90-100)

Prepared using Claude Sonnet 4

In [3]:
# Import required libraries
import pandas as pd
import requests
import time
import json
import os
from tqdm import tqdm
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# Define file paths
DATA_PATH = r"G:\My Drive\Crick_LMS\projects\a_unstudied_genes\priority_unstudied_genes.tsv"
INTERMEDIATE_DIR = r"G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files"
OUTPUT_FILE = os.path.join(INTERMEDIATE_DIR, "plddt_for_PUGs.tsv")

# Create intermediate directory if it doesn't exist
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_FILE}")
print(f"Intermediate directory created/verified: {INTERMEDIATE_DIR}")

Data path: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\priority_unstudied_genes.tsv
Output path: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files\plddt_for_PUGs.tsv
Intermediate directory created/verified: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files


In [5]:
# Load the priority unstudied genes data
print("Loading priority unstudied genes data...")
pugs_df = pd.read_csv(DATA_PATH, sep='\t')

print(f"Loaded {len(pugs_df)} genes from priority unstudied genes dataset")
print(f"Columns: {list(pugs_df.columns)}")
print("\nFirst few rows:")
print(pugs_df.head())

# Extract UniProt IDs
uniprot_ids = pugs_df['UniProt ID'].dropna().unique()
print(f"\nFound {len(uniprot_ids)} unique UniProt IDs")
print("Sample UniProt IDs:", uniprot_ids[:5])

Loading priority unstudied genes data...
Loaded 122 genes from priority unstudied genes dataset
Columns: ['Systematic ID', 'Gene name', 'Product description', 'UniProt ID', 'Synonyms', 'Feature type', 'Start position', 'End position', 'Chromosome', 'Strand', 'Budding yeast orthologs', 'S. japonicus orthologs', 'Human orthologs', 'Physical interactors', 'Deletion viability', 'Transmembrane domain', 'Characterisation status', 'Taxonomic distribution', 'PDB structure IDs', 'Paralog IDs', 'RNACentral ID', 'Protein molecular weight', 'Protein length', 'Protein length (bins)', 'Spliced RNA length', 'Unspliced RNA length']

First few rows:
  Systematic ID Gene name                                Product description  \
0   SPBC1711.12     ppp16  acylpeptide hydrolase, implicated in protein q...   
1   SPCP1E11.10      dhm1    ankyrin repeat protein, unknown biological role   
2  SPBC20F10.03       NaN  armadillo-type fold protein, human IFRD1 ortho...   
3  SPCC16C4.02c      asi1  armadillo-ty

In [30]:
# Define function to fetch PLDDT data from AlphaFold API
def fetch_alphafold_plddt(uniprot_id, max_retries=3, delay=1):
    """
    Fetch PLDDT metrics for a given UniProt ID from AlphaFold API
    
    Parameters:
    uniprot_id (str): UniProt identifier
    max_retries (int): Maximum number of retry attempts
    delay (float): Delay between requests in seconds
    
    Returns:
    dict: Dictionary containing PLDDT metrics or None if failed
    """
    url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            
            if response.status_code == 200:
                data = response.json()
                
                # Extract summary metrics if available
                if data and len(data) > 0:
                    entry = data[0]  # Take first entry.  There is not usually more than one.
                    
                    # Extract PLDDT summary metrics
                    #summary = entry.get('summaryMetrics', {})
                    #plddt_summary = summary.get('plddtMetrics', {})
                    
                    result = {
                        'modelEntityId': entry.get('entryId', uniprot_id),
                        'globalMetricValue': entry.get('globalMetricValue', uniprot_id),
                        'fractionPlddtVeryLow': entry.get('fractionPlddtVeryLow', uniprot_id),
                        'fractionPlddtLow': entry.get('fractionPlddtLow', uniprot_id),
                        'fractionPlddtConfident': entry.get('fractionPlddtConfident', uniprot_id),
                        'fractionPlddtVeryHigh': entry.get('fractionPlddtVeryHigh', uniprot_id)
                    }
                    
                    # Add delay between requests to be respectful
                    time.sleep(delay)
                    return result
                    
            elif response.status_code == 404:
                # No AlphaFold structure available for this UniProt ID
                return {
                    'modelEntityId': uniprot_id,
                    'globalMetricValue': None,
                    'fractionPlddtVeryLow': None,
                    'fractionPlddtLow': None,
                    'fractionPlddtConfident': None,
                    'fractionPlddtVeryHigh': None,
                    'error': 'No AlphaFold structure available'
                }
            else:
                print(f"Attempt {attempt + 1}: HTTP {response.status_code} for {uniprot_id}")
                time.sleep(delay * (attempt + 1))  # Exponential backoff
                
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1}: Network error for {uniprot_id}: {e}")
            time.sleep(delay * (attempt + 1))
    
    # If all attempts failed
    return {
        'modelEntityId': uniprot_id,
        'globalMetricValue': None,
        'fractionPlddtVeryLow': None,
        'fractionPlddtLow': None,
        'fractionPlddtConfident': None,
        'fractionPlddtVeryHigh': None,
        'error': 'API request failed after retries'
    }

print("AlphaFold API function defined successfully!")

AlphaFold API function defined successfully!


In [8]:
# Fetch PLDDT data for all UniProt IDs
print(f"Fetching PLDDT data for {len(uniprot_ids)} UniProt IDs...")
print("This may take several minutes due to API rate limiting.")

plddt_results = []

# Use tqdm for progress tracking
for i, uniprot_id in enumerate(tqdm(uniprot_ids, desc="Fetching PLDDT data")):
    result = fetch_alphafold_plddt(uniprot_id)
    plddt_results.append(result)
    
    # Progress update every 10 entries
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(uniprot_ids)} UniProt IDs")

print(f"\nCompleted fetching data for {len(plddt_results)} entries")

# Convert results to DataFrame
plddt_df = pd.DataFrame(plddt_results)

print(f"Created DataFrame with shape: {plddt_df.shape}")
print(f"Columns: {list(plddt_df.columns)}")
print("\nSample results:")
print(plddt_df.head())

Fetching PLDDT data for 122 UniProt IDs...
This may take several minutes due to API rate limiting.


Fetching PLDDT data:   8%|▊         | 10/122 [00:44<07:56,  4.25s/it]

Processed 10/122 UniProt IDs


Fetching PLDDT data:  16%|█▋        | 20/122 [01:20<06:49,  4.02s/it]

Processed 20/122 UniProt IDs


...

Processed 110/122 UniProt IDs


Fetching PLDDT data:  98%|█████████▊| 120/122 [07:01<00:09,  4.66s/it]

Processed 120/122 UniProt IDs


Fetching PLDDT data: 100%|██████████| 122/122 [07:07<00:00,  3.50s/it]


Completed fetching data for 122 entries
Created DataFrame with shape: (122, 6)
Columns: ['modelEntityId', 'globalMetricValue', 'fractionPlddtVeryLow', 'fractionPlddtLow', 'fractionPlddtConfident', 'fractionPlddtVeryHigh']

Sample results:
  modelEntityId globalMetricValue fractionPlddtVeryLow fractionPlddtLow  \
0  AF-Q9P778-F1              None                 None             None   
1  AF-Q9UU77-F1              None                 None             None   
2  AF-O42973-F1              None                 None             None   
3  AF-O74447-F1              None                 None             None   
4  AF-Q9HDX5-F1              None                 None             None   

  fractionPlddtConfident fractionPlddtVeryHigh  
0                   None                  None  
1                   None                  None  
2                   None                  None  
3                   None                  None  
4                   None                  None  


In [9]:
plddt_df 

,modelEntityId,globalMetricValue,fractionPlddtVeryLow,fractionPlddtLow,fractionPlddtConfident,fractionPlddtVeryHigh
0,AF-Q9P778-F1,None,None,None,None,None
1,AF-Q9UU77-F1,None,None,None,None,None
2,AF-O42973-F1,None,None,None,None,None
3,AF-O74447-F1,None,None,None,None,None
4,AF-Q9HDX5-F1,None,None,None,None,None
...,...,...,...,...,...,...
117,AF-O42948-F1,None,None,None,None,None
118,AF-Q9Y7T2-F1,None,None,None,None,None
119,AF-Q9URW3-F1,None,None,None,None,None
120,AF-Q9HFF2-F1,None,None,None,None,None


In [ ]:
# Analyze and summarize results
print("=== PLDDT Data Analysis Summary ===")

# Basic statistics
print(f"Total entries processed: {len(plddt_df)}")
print(f"Entries with valid PLDDT data: {plddt_df['globalMetricValue'].notna().sum()}")
print(f"Entries without AlphaFold structures: {plddt_df['globalMetricValue'].isna().sum()}")

# Check for any errors
if 'error' in plddt_df.columns:
    error_counts = plddt_df['error'].value_counts()
    print(f"\nError summary:")
    print(error_counts)

# Summary statistics for valid entries
valid_data = plddt_df[plddt_df['globalMetricValue'].notna()]
if len(valid_data) > 0:
    print(f"\n=== Valid PLDDT Statistics ===")
    print(f"Global metric value range: {valid_data['globalMetricValue'].min():.2f} - {valid_data['globalMetricValue'].max():.2f}")
    print(f"Mean global metric value: {valid_data['globalMetricValue'].mean():.2f}")
    
    # Confidence distribution summary
    print(f"\n=== Confidence Distribution (mean values) ===")
    print(f"Very Low confidence (0-50): {valid_data['fractionPlddtVeryLow'].mean():.3f}")
    print(f"Low confidence (50-70): {valid_data['fractionPlddtLow'].mean():.3f}")  
    print(f"Confident (70-90): {valid_data['fractionPlddtConfident'].mean():.3f}")
    print(f"Very High confidence (90-100): {valid_data['fractionPlddtVeryHigh'].mean():.3f}")

print("\nData preview:")
print(plddt_df.head(10))

In [ ]:
# Save results to TSV file
print("Saving results to TSV file...")

# Filter to only the required columns and remove any error columns for final output
output_columns = ['modelEntityId', 'globalMetricValue', 'fractionPlddtVeryLow', 
                 'fractionPlddtLow', 'fractionPlddtConfident', 'fractionPlddtVeryHigh']

# Create output dataframe with only required columns
output_df = plddt_df[output_columns].copy()

# Save to TSV file
output_df.to_csv(OUTPUT_FILE, sep='\t', index=False, na_rep='NA')

print(f"Successfully saved {len(output_df)} entries to: {OUTPUT_FILE}")

# Display final summary
print(f"\n=== Final Output Summary ===")
print(f"File: {OUTPUT_FILE}")
print(f"Shape: {output_df.shape}")
print(f"Columns: {list(output_df.columns)}")

# Show sample of saved data
print(f"\nSample of saved data:")
print(output_df.head(10).to_string())

print(f"\n✅ PLDDT analysis completed successfully!")
print(f"Results saved to: {OUTPUT_FILE}")